In [1]:
from pathlib import Path
from qsiparc.atlas import AtlasLUT, infer_hemisphere, load_lut_from_tsv
from qsiparc.discover import discover_dseg_files, discover_scalar_maps, parse_entities
from qsiparc.extract import extract_scalar_map

In [2]:
from qsiparc.discover import discover_dseg_files, discover_scalar_maps, parse_entities
from qsiparc.extract import extract_scalar_map

In [3]:
qsirecon_dir = Path("/media/storage/yalab-dev/snbb_scheduler/derivatives/qsirecon")

participant_label="S002533"
session_label = "202407110849"

dseg_files = discover_dseg_files(qsirecon_dir=qsirecon_dir, participant_label=participant_label,session_label=session_label, atlas="4S156Parcels")
scalar_maps = discover_scalar_maps(qsirecon_dir=qsirecon_dir, subject=participant_label,session=session_label)

In [4]:
scalar_map = [i for i in scalar_maps if i.entities["param"] == "fa"][0]
dseg_file = dseg_files[0]

In [5]:
from qsiparc.discover import load_lut_for_dseg
lut = load_lut_for_dseg(dseg_file)

In [6]:
res = extract_scalar_map(
    scalar_path=scalar_map.path,
    dseg_path=dseg_file.path,
    lut=lut,
    scalar_name="fa"
)

In [ ]:
def write_diffmap_tsv(
    # df: pd.DataFrame,
    output_dir: Path,
    subject: str,
    session: str,
    atlas_name: str,
    software: str | None = None,
    source_entities: dict[str, str] | None = None,
) -> Path:
    """Write a parcellated diffusion scalar map as a BIDS-derivative TSV.

    Output path::

        <output_dir>/sub-xxx/ses-yyy/dwi/atlas-<atlas_name>/
            sub-xxx_ses-yyy_atlas-<atlas_name>_software-<software>_\\
            <source_entities>_diffmap.tsv

    The ``software`` entity encodes the QSIRecon workflow name (e.g.
    ``"AMICONODDI"`` for ``qsirecon-AMICONODDI``). ``source_entities`` are
    the remaining BIDS entities from the source scalar map file (e.g.
    ``space``, ``model``, ``param``), forwarded in their original order and
    excluding ``sub`` and ``ses`` which are already present at the front.

    Parameters
    ----------
    df : pd.DataFrame
        Long-format DataFrame from `merge_extraction_results`.
    output_dir : Path
        Root of the output derivatives tree.
    subject : str
        Subject label (e.g. "sub-001").
    session : str
        Session label (e.g. "ses-01").
    atlas_name : str
        Atlas name for the directory and filename.
    software : str, optional
        QSIRecon workflow name (e.g. ``"AMICONODDI"``).
    source_entities : dict, optional
        BIDS entities from the source scalar map, forwarded into the filename.
        ``sub`` and ``ses`` are skipped (already encoded in the prefix).

    Returns
    -------
    Path
        Path to the written TSV file.
    """
    atlas_dir = output_dir / subject / session / "dwi" / f"atlas-{atlas_name}"
    # atlas_dir.mkdir(parents=True, exist_ok=True)

    # Build filename: sub_ses_atlas_software_[source entities]_diffmap
    parts = [subject, session, f"atlas-{atlas_name}"]
    if software:
        parts.append(f"software-{software}")
    if source_entities:
        for key, val in source_entities.items():
            if key not in ("sub", "ses"):
                parts.append(f"{key}-{val}")
    parts.append("diffmap")
    filename = "_".join(parts) + ".tsv"

    out_path = atlas_dir / filename

    return out_path

In [ ]:
 write_diffmap_tsv(
                    # df=combined_df,
                    output_dir=Path("outdir"),
                    subject=f"sub-{sub}",
                    session=f"ses-{ses}",
                    atlas_name=atlas_name,
                )

In [7]:
import pandas as pd

def write_diffmap_tsv(
    output_dir: Path,
    subject: str,
    session: str,
    atlas_name: str,
    source_entities: dict[str, str] | None = None,
) -> Path:
    """Write a parcellated diffusion scalar map as a BIDS-derivative TSV.

    Output path:
        <output_dir>/sub-xxx/ses-yyy/dwi/atlas-<atlas_name>/
            sub-xxx_ses-yyy_atlas-<atlas_name>_diffmap.tsv

    Parameters
    ----------
    df : pd.DataFrame
        Long-format DataFrame from `merge_extraction_results`.
    output_dir : Path
        Root of the output derivatives tree.
    subject : str
        Subject label (e.g. "sub-001").
    session : str
        Session label (e.g. "ses-01").
    atlas_name : str
        Atlas name for the directory and filename.
    source_entities : dict, optional
        Additional BIDS entities from the source files to propagate into the filename.

    Returns
    -------
    Path
        Path to the written TSV file.
    """
    atlas_dir = output_dir / subject / session / "dwi" / f"atlas-{atlas_name}"
    atlas_dir.mkdir(parents=True, exist_ok=True)

    # Build filename with optional additional entities
    parts = [subject, session, f"atlas-{atlas_name}"]
    if source_entities:
        for key in ("space", "model", "desc"):
            if key in source_entities:
                parts.append(f"{key}-{source_entities[key]}")
    parts.append("diffmap")
    filename = "_".join(parts) + ".tsv"

    out_path = atlas_dir / filename

    return out_path